# Dont edit this without asking me

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
!df -h

'df' is not recognized as an internal or external command,
operable program or batch file.


In [ ]:
%cd /content/drive/MyDrive/gs_pnr_f1_250325

/content/drive/MyDrive/gear_shop_f2_030325


In [ ]:
%pwd

'/content/drive/MyDrive/gear_shop_f2_030325'

In [ ]:
!unzip ./obj_train_data.zip -d ./

In [ ]:
import shutil
import os

def unzip_using_shutil(zip_file_path, output_folder):
    # Ensure the output folder exists
    os.makedirs(output_folder, exist_ok=True)

    # Use shutil to unpack the archive
    shutil.unpack_archive(zip_file_path, output_folder)
    print(f"Unzipped files into {output_folder}")

# Path to the large zip file and output folder
zip_file_path = '/content/drive/MyDrive/9S_GBA_271225/obj_train_data.zip'
output_folder = '/content/'

unzip_using_shutil(zip_file_path, output_folder)

Unzipped files into /content/


In [ ]:
import os
original_images_path='/content/obj_train_data'  # change this
image_paths = os.listdir(original_images_path)
len(image_paths)

39318

In [ ]:
!pip install --no-deps "pybboxes==0.2.0"

In [ ]:
!pip install imgaug numpy==1.26.4

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 2.6 MB/s eta 0:00:00
INFO: pip is looking at multiple versions of opencv-python to determine which version is compatible with other requirements. This could take a while.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.0/18.0 MB 103.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 948.0/948.0 kB 66.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.0/63.0 MB 40.7 MB/s eta 0:00:00
  Attempting uninstall: numpy
    Found existing installation: numpy 2.0.2
    Uninstalling numpy-2.0.2:
      Successfully uninstalled numpy-2.0.2
  Attempting uninstall: opencv-python
    Found existing installation: opencv-python 4.12.0.88
    Uninstalling opencv-python-4.12.0.88:
      Successfully uninstalled opencv-python-4.12.0.88
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
pybboxes 0.2.0 

In [ ]:
import os  #Edit based on plascticline augmentation
import pybboxes as pbx
import cv2
import imgaug.augmenters as iaa
from imgaug.augmentables.bbs import BoundingBox, BoundingBoxesOnImage
import uuid
from tqdm import tqdm, trange
import random

In [ ]:
import os

# Define the folder path
folder_path = "/content/obj_train_data"

# Iterate over all files in the folder
for file_name in os.listdir(folder_path):
    #print(file_name)
    # Check if the file ends with .PNG (case-insensitive)
    if file_name.lower().endswith(".png"):
        # Get the base name without the .PNG extension
        base_name = os.path.splitext(file_name)[0]

        # Define the corresponding .txt file path
        txt_file_path = os.path.join(folder_path, f"{base_name}.txt")

        # Check if the .txt file exists, if not, create a blank .txt file
        if not os.path.exists(txt_file_path):
            print(f"Creating missing file: {txt_file_path}")
            with open(txt_file_path, 'w') as txt_file:
                pass  # Create an empty file

print("Check completed!")


Creating missing file: /content/obj_train_data/frame_015385.txt
Creating missing file: /content/obj_train_data/frame_015306.txt
Creating missing file: /content/obj_train_data/frame_004380.txt
Creating missing file: /content/obj_train_data/frame_004378.txt
Creating missing file: /content/obj_train_data/frame_015393.txt
Creating missing file: /content/obj_train_data/frame_015266.txt
Creating missing file: /content/obj_train_data/frame_008961.txt
Creating missing file: /content/obj_train_data/frame_015333.txt
Creating missing file: /content/obj_train_data/frame_015358.txt
Creating missing file: /content/obj_train_data/frame_004377.txt
Creating missing file: /content/obj_train_data/frame_015368.txt
Creating missing file: /content/obj_train_data/frame_015269.txt
Creating missing file: /content/obj_train_data/frame_004391.txt
Creating missing file: /content/obj_train_data/frame_009015.txt
Creating missing file: /content/obj_train_data/frame_015349.txt
Creating missing file: /content/obj_trai

In [ ]:
import os
original_images_path='/content/obj_train_data'  # change this
image_paths = os.listdir(original_images_path)
len(image_paths)

39678

In [ ]:
import os
import pybboxes as pbx
import cv2
import imgaug.augmenters as iaa
from imgaug.augmentables.bbs import BoundingBox, BoundingBoxesOnImage
import uuid
from tqdm import tqdm, trange
import random
import time

def convert(size, boxes): # x1,y1,x2,y2,id
  conBbox=[]
  for box in boxes:
    conv=pbx.convert_bbox((box.x1,box.y1,box.x2,box.y2),from_type="voc", to_type="yolo", image_size=size)
    conBbox.append([box.label,conv[0],conv[1],conv[2],conv[3]]) # id,x,y,w,h
  return conBbox

original_images_path='/content/obj_train_data'  # 512
# W=512
# labels=['metal_trolley','plastic_trolley'] # just for name
aug_img = "/content/final_dataset"
os.makedirs(aug_img, exist_ok=True)

pbar = tqdm(total=len(os.listdir(original_images_path))//2)
for file in os.listdir(original_images_path):
    # print(file)
    if file.endswith(".txt"):
        text_lo=os.path.join(original_images_path, file)
        # print(file)
        img=cv2.imread(text_lo[:-4]+".png")
        h,w,_=img.shape
        # print(h,w)
        # if w!=960 and h!=540 :
        #     print("Issue->>>>>",file)
        #     continue
        bbox = []
        with open(text_lo) as f:
            for line in f.readlines():
                l=line.split(" ")
                x1,y1,x2,y2=pbx.convert_bbox((float(l[1]),float(l[2]),float(l[3]),float(l[4])), from_type="yolo", to_type="voc", image_size=(w, h))
                # print(x1,y1,x2,y2)
                bbox.append(BoundingBox(x1=x1, y1=y1, x2=x2, y2=y2,label=int(l[0])))
            bbs = BoundingBoxesOnImage(bbox, shape=img.shape,)
            # print("bbs",bbs)

            values = [0,-4,4]

            # Randomly pick a value
            random_value = random.choice(values)

            for angle in [0]:
                try:
                    seq = iaa.Sequential([
                        iaa.Affine(
                            rotate=angle
                        )
                    ])
                    image_aug, bbs_aug = seq(image=img, bounding_boxes=bbs)
                    bbs_aug=bbs_aug.remove_out_of_image().clip_out_of_image()
                    file_name = str(uuid.uuid4())
                    # print(bbs_aug,"angle")
                    conBbox = convert((w,h), bbs_aug.bounding_boxes)
                    cv2.imwrite(f'{aug_img}/{file_name}.png', image_aug)
                    out_file = open(f'{aug_img}/{file_name}.txt', 'w')
                    for bb in conBbox:
                        out_file.write(" ".join([str(a) for a in bb]) + '\n')
                    out_file.close()
                    # img_after = bbs_aug.draw_on_image(image_aug, size=2, color=[0, 0, 255])
                    # cv2.imshow(f"ang",img_after)
                    # cv2.waitKey(0)
                    # break
                except:
                    continue


                # for bright in [-20]:
                #     seq = iaa.Sequential([
                #         iaa.Affine(
                #             rotate=angle
                #         ),
                #         iaa.AddToBrightness(bright)
                #     ])
                #     image_aug, bbs_aug = seq(image=img, bounding_boxes=bbs)
                #     bbs_aug=bbs_aug.remove_out_of_image().clip_out_of_image()
                #     file_name = str(uuid.uuid4())
                #     # print(bbs_aug,"bright")
                #     conBbox = convert((w,h), bbs_aug.bounding_boxes)
                #     cv2.imwrite(f'{aug_img}/{file_name}.png',image_aug)
                #     out_file = open(f'{aug_img}/{file_name}.txt', 'w')
                #     for bb in conBbox:
                #         out_file.write(" ".join([str(a) for a in bb]) + '\n')
                #     out_file.close()
                    # img_after = bbs_aug.draw_on_image(image_aug, size=2, color=[0, 0, 255])
                    # cv2.imshow(f"{bright}",img_after)
                    # cv2.waitKey(0)

                # for hue in [-15,15]:
#                     seq = iaa.Sequential([
#                         iaa.Affine(
#                             rotate=angle
#                         ),
#                         iaa.AddToHue(hue)

#                     ])
#                     image_aug, bbs_aug = seq(image=img, bounding_boxes=bbs)
#                     bbs_aug=bbs_aug.remove_out_of_image().clip_out_of_image()
#                     file_name = str(uuid.uuid4())
#                     # print(bbs_aug)
#                     conBbox = convert((w,h), bbs_aug.bounding_boxes)
#                     cv2.imwrite(f'{aug_img}/{file_name}.png',image_aug)
#                     out_file = open(f'{aug_img}/{file_name}.txt', 'w')
#                     for bb in conBbox:
#                         out_file.write(" ".join([str(a) for a in bb]) + '\n')
#                     out_file.close()
#                     # img_after = bbs_aug.draw_on_image(image_aug, size=2, color=[0, 0, 255])
#                     # cv2.imshow(f"{hue}",img_after)
#                     # cv2.waitKey(0)


#                 for satu in [-10,10]:
#                     seq = iaa.Sequential([
#                         iaa.Affine(
#                             rotate=angle
#                         ),
#                         iaa.AddToSaturation(satu)

#                     ])
#                     image_aug, bbs_aug = seq(image=img, bounding_boxes=bbs)
#                     bbs_aug=bbs_aug.remove_out_of_image().clip_out_of_image()
#                     file_name = str(uuid.uuid4())
#                     # print(bbs_aug)
#                     conBbox = convert((w,h), bbs_aug.bounding_boxes)
#                     cv2.imwrite(f'{aug_img}/{file_name}.png',image_aug)
#                     out_file = open(f'{aug_img}/{file_name}.txt', 'w')
#                     for bb in conBbox:
#                         out_file.write(" ".join([str(a) for a in bb]) + '\n')
#                     out_file.close()
#                     # img_after = bbs_aug.draw_on_image(image_aug, size=2, color=[0, 0, 255])
#                     # cv2.imshow(f"{satu}",img_after)
#                     # cv2.waitKey(0)

#                 # equalize augment
#                 seq = iaa.Sequential([
#                     iaa.Affine(
#                         rotate=angle
#                     ),
#                     iaa.pillike.Equalize()
#                 ])
#                 image_aug, bbs_aug = seq(image=img, bounding_boxes=bbs)
#                 bbs_aug=bbs_aug.remove_out_of_image().clip_out_of_image()
#                 file_name = str(uuid.uuid4())
#                 # print(bbs_aug)
#                 conBbox = convert((w, h), bbs_aug.bounding_boxes)
#                 cv2.imwrite(f'{aug_img}/{file_name}.png',image_aug)
#                 out_file = open(f'{aug_img}/{file_name}.txt', 'w')
#                 for bb in conBbox:
#                     out_file.write(" ".join([str(a) for a in bb]) + '\n')
#                 out_file.close()
#                 # img_after = bbs_aug.draw_on_image(image_aug, size=2, color=[0, 0, 255])
#                 # cv2.imshow(f"equalize",img_after)
#                 # cv2.waitKey(0)

                # solarize augment
                # for sol in [0.4,0.5,0.6]:
                #     seq = iaa.Sequential([
                #         iaa.Affine(
                #             rotate=angle
                #         ),
                #         iaa.pillike.Solarize(p=sol)
                #     ])
                #     image_aug, bbs_aug = seq(image=img, bounding_boxes=bbs)
                #     bbs_aug=bbs_aug.remove_out_of_image().clip_out_of_image()
                #     file_name = str(uuid.uuid4())
                #     # print(bbs_aug)
                #     conBbox = convert((w, h), bbs_aug.bounding_boxes)
                #     cv2.imwrite(f'{aug_img}/{file_name}.png',image_aug)
                #     out_file = open(f'{aug_img}/{file_name}.txt', 'w')
                #     for bb in conBbox:
                #         out_file.write(" ".join([str(a) for a in bb]) + '\n')
                #     out_file.close()
                #     # img_after = bbs_aug.draw_on_image(image_aug, size=2, color=[0, 0, 255])
                #     # cv2.imshow(f"{sol}",img_after)
                #     # cv2.waitKey(0)

#                 # EnhanceColor
#                 for enh in [0.8]:
#                     seq = iaa.Sequential([
#                         iaa.Affine(
#                             rotate=angle
#                         ),
#                         iaa.pillike.EnhanceColor(factor=enh)

#                     ])
#                     image_aug, bbs_aug = seq(image=img, bounding_boxes=bbs)
#                     bbs_aug=bbs_aug.remove_out_of_image().clip_out_of_image()
#                     file_name = str(uuid.uuid4())
#                     # print(bbs_aug)
#                     conBbox = convert((w, h), bbs_aug.bounding_boxes)
#                     cv2.imwrite(f'{aug_img}/{file_name}.png',image_aug)
#                     out_file = open(f'{aug_img}/{file_name}.txt', 'w')
#                     for bb in conBbox:
#                         out_file.write(" ".join([str(a) for a in bb]) + '\n')
#                     out_file.close()
#                     # img_after = bbs_aug.draw_on_image(image_aug, size=2, color=[0, 0, 255])
#                     # cv2.imshow(f"{enh}",img_after)
#                     # cv2.waitKey(0)

#                 # Enhance Sharpness
#                 for enhsh in [1.4]:
#                     seq = iaa.Sequential([
#                         iaa.Affine(
#                             rotate=angle
#                         ),
#                         iaa.pillike.EnhanceSharpness(factor=enhsh)

#                     ])
#                     image_aug, bbs_aug = seq(image=img, bounding_boxes=bbs)
#                     bbs_aug=bbs_aug.remove_out_of_image().clip_out_of_image()
#                     file_name = str(uuid.uuid4())
#                     # print(bbs_aug)
#                     conBbox = convert((w, h), bbs_aug.bounding_boxes)
#                     cv2.imwrite(f'{aug_img}/{file_name}.png',image_aug)
#                     out_file = open(f'{aug_img}/{file_name}.txt', 'w')
#                     for bb in conBbox:
#                         out_file.write(" ".join([str(a) for a in bb]) + '\n')
#                     out_file.close()
#                     # img_after = bbs_aug.draw_on_image(image_aug, size=2, color=[0, 0, 255])
#                     # cv2.imshow(f"{enhsh}",img_after)
#                     # cv2.waitKey(0)

                # Translation
                # try:
                #     for y in range(25,52,25):
                #         seq = iaa.Sequential([
                #             iaa.Affine(
                #                 rotate=angle
                #             ),
                #             iaa.Affine(translate_px={"x": 0, "y": y})
                #         ])
                #         image_aug, bbs_aug = seq(image=img, bounding_boxes=bbs)
                #         bbs_aug=bbs_aug.remove_out_of_image().clip_out_of_image()
                #         file_name = str(uuid.uuid4())
                #         # print(bbs_aug)
                #         conBbox = convert((w,h), bbs_aug.bounding_boxes)
                #         cv2.imwrite(f'{aug_img}/{file_name}.png', image_aug)
                #         out_file = open(f'{aug_img}/{file_name}.txt', 'w')
                #         for bb in conBbox:
                #             out_file.write(" ".join([str(a) for a in bb]) + '\n')
                #         out_file.close()
                #         # img_after = bbs_aug.draw_on_image(image_aug, size=2, color=[0, 0, 255])
                #         # cv2_imshow(img_after)
                #         # cv2.waitKey(0)
                # except:
                #     pass
                # try:
                #     for y in range(25,52,25):
                #         seq = iaa.Sequential([
                #             iaa.Affine(
                #                 rotate=angle
                #             ),
                #             iaa.Affine(translate_px={"x": 0, "y": -y})
                #         ])
                #         image_aug, bbs_aug = seq(image=img, bounding_boxes=bbs)
                #         bbs_aug=bbs_aug.remove_out_of_image().clip_out_of_image()
                #         file_name = str(uuid.uuid4())
                #         # print(bbs_aug)
                #         conBbox = convert((w,h), bbs_aug.bounding_boxes)
                #         cv2.imwrite(f'{aug_img}/{file_name}.png', image_aug)
                #         out_file = open(f'{aug_img}/{file_name}.txt', 'w')
                #         for bb in conBbox:
                #             out_file.write(" ".join([str(a) for a in bb]) + '\n')
                #         out_file.close()
                #         # img_after = bbs_aug.draw_on_image(image_aug, size=2, color=[0, 0, 255])
                #         # cv2_imshow(img_after)
                #         # cv2.waitKey(0)
                # except:
                #     pass

                # try:
                #     for x in range(25,26,25):
                #         seq = iaa.Sequential([
                #             iaa.Affine(
                #                 rotate=angle
                #             ),
                #             iaa.Affine(translate_px={"x": x, "y": 0})
                #         ])
                #         image_aug, bbs_aug = seq(image=img, bounding_boxes=bbs)
                #         bbs_aug=bbs_aug.remove_out_of_image().clip_out_of_image()
                #         file_name = str(uuid.uuid4())
                #         # print(bbs_aug,"+x")
                #         conBbox = convert((w,h), bbs_aug.bounding_boxes)
                #         cv2.imwrite(f'{aug_img}/{file_name}.png', image_aug)
                #         out_file = open(f'{aug_img}/{file_name}.txt', 'w')
                #         for bb in conBbox:
                #             out_file.write(" ".join([str(a) for a in bb]) + '\n')
                #         out_file.close()
                #         # img_after = bbs_aug.draw_on_image(image_aug, size=2, color=[0, 0, 255])
                #         # cv2_imshow(img_after)
                #         # cv2.waitKey(0)
                # except:
                #     pass
                # try:
                #     for x in range(25,52,25):
                #         seq = iaa.Sequential([
                #             iaa.Affine(
                #                 rotate=angle
                #             ),
                #             iaa.Affine(translate_px={"x": -x, "y": 0})
                #         ])
                #         image_aug, bbs_aug = seq(image=img, bounding_boxes=bbs)
                #         bbs_aug=bbs_aug.remove_out_of_image().clip_out_of_image()
                #         file_name = str(uuid.uuid4())
                #         # print(bbs_aug,"-x")
                #         conBbox = convert((w,h), bbs_aug.bounding_boxes)
                #         cv2.imwrite(f'{aug_img}/{file_name}.png', image_aug)
                #         out_file = open(f'{aug_img}/{file_name}.txt', 'w')
                #         for bb in conBbox:
                #             out_file.write(" ".join([str(a) for a in bb]) + '\n')
                #         out_file.close()
                #         # img_after = bbs_aug.draw_on_image(image_aug, size=2, color=[0, 0, 255])
                #         # cv2_imshow(img_after)
                #         # cv2.waitKey(0)
                # except:
                #     pass
                # try:
                #     for y in range(25,52,25):
                #         seq = iaa.Sequential([
                #             iaa.Affine(
                #                 rotate=angle
                #             ),
                #             iaa.Affine(translate_px={"x": 0, "y": y})
                #         ])
                #         image_aug, bbs_aug = seq(image=img, bounding_boxes=bbs)
                #         bbs_aug=bbs_aug.remove_out_of_image().clip_out_of_image()
                #         file_name = str(uuid.uuid4())
                #         # print(bbs_aug,"+x")
                #         conBbox = convert((w,h), bbs_aug.bounding_boxes)
                #         cv2.imwrite(f'{aug_img}/{file_name}.png', image_aug)
                #         out_file = open(f'{aug_img}/{file_name}.txt', 'w')
                #         for bb in conBbox:
                #             out_file.write(" ".join([str(a) for a in bb]) + '\n')
                #         out_file.close()
                #         # img_after = bbs_aug.draw_on_image(image_aug, size=2, color=[0, 0, 255])
                #         # cv2_imshow(img_after)
                #         # cv2.waitKey(0)
                # except:
                #     pass
                try:
                    for y in range(25,26,25):
                        seq = iaa.Sequential([
                            iaa.Affine(
                                rotate=angle
                            ),
                            iaa.Affine(translate_px={"x": 0, "y": -y})
                        ])
                        image_aug, bbs_aug = seq(image=img, bounding_boxes=bbs)
                        bbs_aug=bbs_aug.remove_out_of_image().clip_out_of_image()
                        file_name = str(uuid.uuid4())
                        # print(bbs_aug,"-x")
                        conBbox = convert((w,h), bbs_aug.bounding_boxes)
                        cv2.imwrite(f'{aug_img}/{file_name}.png', image_aug)
                        out_file = open(f'{aug_img}/{file_name}.txt', 'w')
                        for bb in conBbox:
                            out_file.write(" ".join([str(a) for a in bb]) + '\n')
                        out_file.close()
                        # img_after = bbs_aug.draw_on_image(image_aug, size=2, color=[0, 0, 255])
                        # cv2_imshow(img_after)
                        # cv2.waitKey(0)
                except:
                    pass
                # seq = iaa.Sequential([
                #         iaa.Affine(
                #             rotate=angle
                #         ),
                #         iaa.Fliplr(1)
                #     ])
                # image_aug, bbs_aug = seq(image=img, bounding_boxes=bbs)
                # bbs_aug=bbs_aug.remove_out_of_image().clip_out_of_image()
                # file_name = str(uuid.uuid4())
                # # print(bbs_aug,"lr")
                # conBbox = convert((w, h), bbs_aug.bounding_boxes)
                # cv2.imwrite(f'{aug_img}/{file_name}.png',image_aug)
                # out_file = open(f'{aug_img}/{file_name}.txt', 'w')
                # for bb in conBbox:
                #     out_file.write(" ".join([str(a) for a in bb]) + '\n')
                # out_file.close()
                #img_after = bbs_aug.draw_on_image(image_aug, size=2, color=[0, 0, 255])
                #print("After aug", "equalize")
                #cv2_imshow(img_after)
                # cv2.imshow(f"equalize",img_after)
                # cv2.waitKey(0)

                # seq = iaa.Sequential([
                #         iaa.Affine(
                #             rotate=angle
                #         ),
                #         iaa.Flipud(1)
                #     ])
                # image_aug, bbs_aug = seq(image=img, bounding_boxes=bbs)
                # bbs_aug=bbs_aug.remove_out_of_image().clip_out_of_image()
                # file_name = str(uuid.uuid4())
                # # print(bbs_aug,"ud")
                # conBbox = convert((w, h), bbs_aug.bounding_boxes)
                # cv2.imwrite(f'{aug_img}/{file_name}.png',image_aug)
                # out_file = open(f'{aug_img}/{file_name}.txt', 'w')
                # for bb in conBbox:
                #     out_file.write(" ".join([str(a) for a in bb]) + '\n')
                # out_file.close()
                #img_after = bbs_aug.draw_on_image(image_aug, size=2, color=[0, 0, 255])
                #print("After aug", "equalize")
                #cv2_imshow(img_after)
                # cv2.imshow(f"equalize",img_after)
                # cv2.waitKey(0)

                # # Grey color convertion
                # gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
                # converted_img = cv2.cvtColor(gray, cv2.COLOR_GRAY2BGR)
                # file_name = str(uuid.uuid4())
                # cv2.imwrite(f'{aug_img}/{file_name}.png', converted_img)
                # #Copy original text file same data in new file
                # out_file = open(f'{aug_img}/{file_name}.txt', 'w')
                # with open(text_lo) as DF:
                #     for line in DF.readlines():
                #         out_file.write(line)
                # out_file.close()


        pbar.update(1)
                # break

        # break

pbar.close()
# cv2.destroyAllWindows()
time.sleep(3)

100%|██████████| 19839/19839 [1:16:48<00:00,  4.31it/s]


In [ ]:
import os
images_path = '/content/final_dataset'  # change this
image_p = os.listdir(images_path)
len(image_p)

79356

In [ ]:
# ⚠ ⚠ ⚠ ⚠ ⚠ ⚠ ⚠ ⚠
import os
import glob

files = glob.glob('./obj_train_data/*')
# print(files)
for f in files:
    print(f)
    os.remove(f)

In [ ]:
# ⚠ ⚠ ⚠ ⚠ ⚠ ⚠ ⚠ ⚠
import os
import glob

files = glob.glob('./final_dataset/*')
# print(files)
for f in files:
    print(f)
    os.remove(f)

In [ ]:
# yolov8 format
output_dir = "/content/train_dataset"
os.makedirs(output_dir, exist_ok=True)
os.makedirs(output_dir+'/train', exist_ok=True)
os.makedirs(output_dir+'/train/images', exist_ok=True)
os.makedirs(output_dir+'/train/labels', exist_ok=True)
os.makedirs(output_dir+'/valid', exist_ok=True)
os.makedirs(output_dir+'/valid/images', exist_ok=True)
os.makedirs(output_dir+'/valid/labels', exist_ok=True)

In [ ]:
import random
import glob
image_paths = glob.glob('/content/final_dataset/*.png')
# image_paths = glob.glob('./obj_train_data/*.png')
random.shuffle(image_paths)
print(len(image_paths))
train_data = image_paths[:int((len(image_paths)+1)*0.70)]
val_data = image_paths[int((len(image_paths)+1)*0.70):]
print(len(train_data), len(val_data))

39678
27775 11903


In [ ]:
import shutil
from tqdm import tqdm, trange
pbar = tqdm(total=len(train_data)+len(val_data))
for paths in train_data:
  # print(paths.split("\\")[-1])
  name = paths.split('/')[-1]
  shutil.move(paths, f"{output_dir}/train/images/{name}")
  shutil.move(f"{paths[:-4]}.txt", f"{output_dir}/train/labels/{name[:-4]}.txt")
  pbar.update(1)
print("train done")
for paths in val_data:
  # print(paths.split("/")[-1])
  name = paths.split('/')[-1]
  shutil.move(paths, f"{output_dir}/valid/images/{name}")
  shutil.move(f"{paths[:-4]}.txt", f"{output_dir}/valid/labels/{name[:-4]}.txt")
  pbar.update(1)
print("test done")

pbar.close()

 79%|███████▊  | 31243/39678 [00:01<00:00, 20541.81it/s]

train done


100%|██████████| 39678/39678 [00:01<00:00, 20848.36it/s]

test done


# Yolov11

In [ ]:
%cd /content/drive/MyDrive/9S_GBA_271225

/content/drive/.shortcut-targets-by-id/13myZnpEJ6PSpdfD8apvpIwd7b4AqxE3T/9S_GBA_271225


In [ ]:
!pip install ultralytics

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 22.4 MB/s eta 0:00:00


In [ ]:
import torch
torch.cuda.empty_cache()


In [12]:
from ultralytics import YOLO
model = YOLO('yolo12x.pt')
results = model.train(data='/content/drive/MyDrive/9S_GBA_271225/dfyolo.yaml', epochs=25, imgsz=640,batch=2,device=[0],close_mosaic = 0)  # ,save_period = 5

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
Ultralytics 8.3.241 🚀 Python-3.12.12 torch-2.9.0+cu126 CUDA:0 (NVIDIA A100-SXM4-40GB, 40507MiB)
engine/trainer: agnostic_nms=False, amp=True, augment=False, auto_augment=randaugment, batch=2, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=0, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/drive/MyDrive/9S_GBA_271225/dfyolo.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=25, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, 

In [ ]:
### GREENY ###

In [ ]:
#checking old model
from ultralytics import YOLO
mcb_model = YOLO('/content/drive/MyDrive/gs_pnr_f1_250325/bestF1.pt', task="detect",)
print(mcb_model.names)

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
{0: 'gear_box', 1: 'engine_block', 2: 'aligner', 3: 'locking_lever', 4: 'tightening_tool', 5: 'gear_box_black'}
